In [1]:
import os
import pandas as pd
from lxml import etree

In [2]:
year="2025"
empresa="RGA"
# Especificar el directorio de los archivos XML
directory_path = rf'C:\Users\RGARCIA\Downloads\SAT 2020\Operaciones\2026 XML\Recibidos\Trabajo Ingresos'
d_path = rf'C:\Users\RGARCIA\Downloads\SAT 2020\Operaciones\2026 XML\Recibidos'

entrada= "R" # Seleccion de emitidos (1) o recibidos (0)

if entrada == "E":
    sentido = "Emitidos"
elif entrada == "R":
    sentido = "Recibidos"
else:
    sentido = "Valor inválido"

def clean_xml(file_path):
    """Elimina xmlns:schemaLocation del XML antes de procesarlo."""
    with open(file_path, "r", encoding="utf-8") as f:
        xml_content = f.read()

    # Remover el atributo xmlns:schemaLocation del XML
    xml_content = xml_content.replace('xmlns:schemaLocation', 'xmlns')

    return xml_content
def list_files_in_directory(directory):
    file_list = []
    
    for root, dirs, files in os.walk(directory):
        for file in files:
#            if file.endswith('.xml'):
            if file.lower().endswith('.xml'):
                file_path = os.path.join(root, file)

                try:
                    # Limpiar el XML antes de parsearlo
                    cleaned_xml = clean_xml(file_path)

                    # Usar un parser que permita recuperación de errores
                    parser = etree.XMLParser(recover=True)
                    tree = etree.fromstring(cleaned_xml.encode(), parser)
                    root_element = tree
                    
                    version = root_element.get("Version", root_element.get("version"))
                    
                    if version == "3.2":
                        namespaces = {
                            'cfdi': 'http://www.sat.gob.mx/cfd/3',
                            'tfd': 'http://www.sat.gob.mx/TimbreFiscalDigital',
                            'implocal': 'http://www.sat.gob.mx/implocal'  
                        }
                    elif version == "3.3":
                        namespaces = {
                            'cfdi': 'http://www.sat.gob.mx/cfd/3',
                            'cce20': 'http://www.sat.gob.mx/ComercioExterior20',
                            'tfd': 'http://www.sat.gob.mx/TimbreFiscalDigital',
                            'implocal': 'http://www.sat.gob.mx/implocal'  
                        }
                    elif version == "4.0":
                        namespaces = {
                            'cfdi': 'http://www.sat.gob.mx/cfd/4',
                            'cce20': 'http://www.sat.gob.mx/ComercioExterior20',
                            'tfd': 'http://www.sat.gob.mx/TimbreFiscalDigital',
                            'implocal': 'http://www.sat.gob.mx/implocal'  
                        }
                    else:
                        print(f"Versión de CFDI desconocida en archivo: {file_path}")
                        continue

                    metodo_pago = root_element.get('MetodoPago', 'N/A')
                    forma_pago = root_element.get('FormaPago', 'N/A')
                    subtotald = root_element.get('SubTotal', 0)
                    descuentod = root_element.get('Descuento', 0)
                    totald = root_element.get('Total', 0)
                    tipoc = root_element.get('TipoDeComprobante', 0)
                    moneda = root_element.get('Moneda', 'N/A')
                    tipo_cambio = root_element.get('TipoCambio', 'N/A')
                    ffecha = root_element.get('Fecha', 'N/A')
                    ffolio = root_element.get('Folio', 'N/A')
                    fserie = root_element.get('Serie', 'N/A')
                    fexporta = root_element.get('Exportacion', 'N/A')
                    fcondiciones = root_element.get('CondicionesDePago', 'N/A')

                    receptord = root_element.find('.//cfdi:Receptor', namespaces)
                    emisord = root_element.find('.//cfdi:Emisor', namespaces)

                    rfce = emisord.get('Rfc', 'N/A') if emisord is not None else 'N/A'
                    nombree = emisord.get('Nombre', 'N/A') if emisord is not None else 'N/A'
                    rege = emisord.get('RegimenFiscal', 'N/A') if emisord is not None else 'N/A'

                    rfcr = receptord.get('Rfc', 'N/A') if receptord is not None else 'N/A'
                    nombrer = receptord.get('Nombre', 'N/A') if receptord is not None else 'N/A'
                    regr = receptord.get('RegimenFiscalReceptor', 'N/A') if receptord is not None else 'N/A'
                    usore= receptord.get('UsoCFDI', 'N/A') if receptord is not None else 'N/A'

                    conceptos = root_element.findall('.//cfdi:Conceptos/cfdi:Concepto', namespaces)
                    
                    timbre_fiscal = root_element.find('.//tfd:TimbreFiscalDigital', namespaces)
                    uuid = timbre_fiscal.get('UUID', 'N/A') if timbre_fiscal is not None else 'N/A'

                    # Buscar impuestos locales (manejo de error si no existe el prefijo)
                    complemento = tree.find('.//cfdi:Complemento', namespaces)
                    impuestos_locales = None
                    traslados_locales = []

                    for concepto in conceptos:
                        ClaveP_S = concepto.get('ClaveProdServ', 'N/A')
                        Descrip_ps = concepto.get('Descripcion', 'N/A')
                        Qtty = concepto.get('Cantidad', 'N/A')
                        UM = concepto.get('Unidad', 'N/A')
                        ValorU = concepto.get('ValorUnitario', 'N/A')
                        Monto = concepto.get('Importe', 'N/A')
                        descuentoi = concepto.get('Descuento', 0)
                        
                        file_list.append({
                            "Archivo": file,
                            "UUID": uuid,
                            "Subtotal": subtotald,
                            "Descuento": descuentod,
                            "Total": totald,
                            "TipoC": tipoc,
                            "Metodo de Pago": metodo_pago,
                            "Forma de Pago": forma_pago,
                            "Moneda": moneda,
                            "Tipo Cambio": tipo_cambio,
                            "Fecha": ffecha,
                            "Folio": ffolio,
                            "Serie": fserie,
                            "Exportacion": fexporta,
                            "Condiciones de pago": fcondiciones,
                            "RFC Recep": rfcr,
                            "Nombre Recep": nombrer,
                            "Regimen Fiscal Rec": regr,
                            "Uso Comp": usore,
                            "RFC Emisor": rfce,
                            "Nombre Emisor": nombree,
                            "Regimen Fiscal Emi": rege,
                            "Clave_Prod": ClaveP_S,
                            "Descripcion": Descrip_ps,
                            "Cantidad": Qtty,
                            "Unidad M": UM,
                            "ValorU": ValorU,
                            "Importe": Monto,
                            "DescuentoC": descuentoi,
                            "Base": '',
                            "Impuesto": '',
                            "Importe Impositivo": '',
                            "Tipo_Factor": '',
                            "Tipo": "C"  # Indica que es un concepto
                        })


                        
                        impuestos = concepto.find('.//cfdi:Impuestos', namespaces)
                        traslados = impuestos.findall('.//cfdi:Traslado', namespaces) if impuestos is not None else []
                        retenciones = impuestos.findall('.//cfdi:Retencion', namespaces) if impuestos is not None else []
                        


                        for traslado in traslados:
                            basei = traslado.get('Base', 0)
                            clavei = traslado.get('Impuesto', 0)
                            importei = traslado.get('Importe', 0)
                            tipofi = traslado.get('TipoFactor', 0)

                            file_list.append({
                                "Archivo": file,
                                "UUID": uuid,
                                "Subtotal": subtotald,
                                "Descuento": descuentod,
                                "Total": totald,
                                "TipoC": tipoc,
                                "Metodo de Pago": metodo_pago,
                                "Forma de Pago": forma_pago,
                                "Moneda": moneda,
                                "Tipo Cambio": tipo_cambio,
                                "Fecha": ffecha,
                                "Folio": ffolio,
                                "Serie": fserie,
                                "Exportacion": fexporta,
                                "Condiciones de pago": fcondiciones,
                                "RFC Recep": rfcr,
                                "Nombre Recep": nombrer,
                                "Regimen Fiscal Rec": regr,
                                "Uso Comp": usore,
                                "RFC Emisor": rfce,
                                "Nombre Emisor": nombree,
                                "Regimen Fiscal Emi": rege,
                                "Clave_Prod": ClaveP_S,
                                "Descripcion": Descrip_ps,
                                "Cantidad": Qtty,
                                "Unidad M": UM,
                                "ValorU": '',
                                "Importe": importei,
                                "DescuentoC": '',
                                "Base": basei,
                                "Impuesto": clavei,
                                "Importe Impositivo": '',
                                "Tipo_Factor": tipofi,
                                "Tipo": "T"  # Indica que es un traslado
                            })


                        # Extraer retenciones (si existen)
                        for retencion in retenciones:
                            baser = retencion.get('Base', 0)
                            claver = retencion.get('Impuesto', 0)
                            importer = retencion.get('Importe', 0)
                            tipofr = retencion.get('TipoFactor', 0)

                            # Agregar retención a la lista
                            file_list.append({
                                "Archivo": file,
                                "UUID": uuid,
                                "Subtotal": subtotald,
                                "Descuento": descuentod,
                                "Total": totald,
                                "TipoC": tipoc,
                                "Metodo de Pago": metodo_pago,
                                "Forma de Pago": forma_pago,
                                "Moneda": moneda,
                                "Tipo Cambio": tipo_cambio,
                                "Fecha": ffecha,
                                "Folio": ffolio,
                                "Serie": fserie,
                                "Exportacion": fexporta,
                                "Condiciones de pago": fcondiciones,
                                "RFC Recep": rfcr,
                                "Nombre Recep": nombrer,
                                "Regimen Fiscal Rec": regr,
                                "Uso Comp": usore,
                                "Regimen Fiscal Rec": regr,
                                "RFC Emisor": rfce,
                                "Nombre Emisor": nombree,
                                "Regimen Fiscal Emi": rege,
                                "Clave_Prod": ClaveP_S,
                                "Descripcion": Descrip_ps,
                                "Cantidad": Qtty,
                                "Unidad M": UM,
                                "ValorU": '',
                                "Importe": importer,
                                "DescuentoC": '',
                                "Base": baser,
                                "Impuesto": claver,
                                "Importe Impositivo": '',
                                "Tipo_Factor": tipofr,
                                "Tipo": "R"  # Indica que es una retención
                            })
                    try:
                        impuestos_locales = complemento.find('.//implocal:ImpuestosLocales', namespaces) if complemento is not None else None
                        traslados_locales = impuestos_locales.findall('.//implocal:TrasladosLocales', namespaces) if impuestos_locales is not None else []
                    except KeyError:
                        # Si no existe el prefijo implocal, no hacer nada
                        traslados_locales = []
                    # Procesar impuestos locales
                    for traslado_local in traslados_locales:
                        imp_local = traslado_local.get('ImpLocTrasladado', '')
                        tasa_local = traslado_local.get('TasadeTraslado', 0)
                        importe_local = traslado_local.get('Importe', 0)
                    
                        file_list.append({
                            "Archivo": file,
                            "UUID": uuid,
                            "Subtotal": subtotald,
                            "Descuento": descuentod,
                            "Total": totald,
                            "TipoC": tipoc,
                            "Metodo de Pago": metodo_pago,
                            "Forma de Pago": forma_pago,
                            "Moneda": moneda,
                            "Tipo Cambio": tipo_cambio,
                            "Fecha": ffecha,
                            "Folio": ffolio,
                            "Serie": fserie,
                            "Exportacion": fexporta,                            
                            "Condiciones de pago": fcondiciones,
                            "RFC Recep": rfcr,
                            "Nombre Recep": nombrer,
                            "Regimen Fiscal Rec": regr,
                            "Uso Comp": usore,
                            "RFC Emisor": rfce,
                            "Nombre Emisor": nombree,
                            "Regimen Fiscal Emi": rege,
                            "Clave_Prod": ClaveP_S,
                            "Descripcion": Descrip_ps,
                            "Cantidad": Qtty,
                            "Unidad M": UM,
                            "ValorU": '',
                            "Importe": importe_local,
                            "DescuentoC": '',
                            "Base": '',
                            "Impuesto": imp_local,
                            "Importe Impositivo": '',







                            
                            "Tipo_Factor": tasa_local,
                            "Tipo": "L"
                        })                        
                except etree.XMLSyntaxError:
                    print(f"Invalid XML in file: {file_path}")
                except Exception as e:
                    print(f"Error processing file {file_path}: {e}")

    df = pd.DataFrame(file_list)
    return df



#-#directory_path = rf'C:\Users\RGARCIA\Downloads\Arzate\CFDI 2024'
#-#d_path = rf'C:\Users\RGARCIA\Downloads\Arzate'
df_files = list_files_in_directory(directory_path)    


In [3]:
# Definir la ruta completa para el archivo CSV de salida
output_csv_path = os.path.join(d_path, f'{sentido} {empresa}.csv')

# Guardar el DataFrame ordenado en la ruta especificada
df_files.to_csv(output_csv_path,index = False)

print(f"✅ El archivo se ha guardado en: {output_csv_path}")

✅ El archivo se ha guardado en: C:\Users\RGARCIA\Downloads\SAT 2020\Operaciones\2026 XML\Recibidos\Recibidos RGA.csv
